# Model Building and Experiment Tracking

This notebook develops machine learning models to predict whether a customer
will purchase the Wellness Tourism Package.

The processed training and testing datasets are loaded directly from the
Hugging Face Dataset repository.

The model development process includes:
- Feature preprocessing
- Model training
- Hyperparameter tuning
- Model evaluation
- Experiment tracking using MLflow
- Selection of the best-performing model

In [72]:
# Import data handling, experiment tracking, preprocessing, model, and evaluation utilities.
import pandas as pd
import numpy as np

import mlflow
import mlflow.sklearn
import os
import skops.io as sio

from datasets import load_dataset

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from huggingface_hub import HfApi
from huggingface_hub import hf_hub_download


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

In [ ]:
# Load the versioned train and test splits from the Hugging Face dataset repository.
repo_id = "motidev/wellness-tourism-dataset"

dataset = load_dataset(
    "csv",
    data_files={
        "train": f"hf://datasets/{repo_id}/processed/train.csv",
        "test": f"hf://datasets/{repo_id}/processed/test.csv"
    }
)

dataset

DatasetDict({
    train: Dataset({
        features: ['Age', 'TypeofContact', 'CityTier', 'DurationOfPitch', 'Occupation', 'Gender', 'NumberOfPersonVisiting', 'NumberOfFollowups', 'ProductPitched', 'PreferredPropertyStar', 'MaritalStatus', 'NumberOfTrips', 'Passport', 'PitchSatisfactionScore', 'OwnCar', 'NumberOfChildrenVisiting', 'Designation', 'MonthlyIncome', 'ProdTaken'],
        num_rows: 3302
    })
    test: Dataset({
        features: ['Age', 'TypeofContact', 'CityTier', 'DurationOfPitch', 'Occupation', 'Gender', 'NumberOfPersonVisiting', 'NumberOfFollowups', 'ProductPitched', 'PreferredPropertyStar', 'MaritalStatus', 'NumberOfTrips', 'Passport', 'PitchSatisfactionScore', 'OwnCar', 'NumberOfChildrenVisiting', 'Designation', 'MonthlyIncome', 'ProdTaken'],
        num_rows: 826
    })
})

In [ ]:
# Convert each Hugging Face split to pandas and verify the expected row counts.
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

Training shape: (3302, 19)
Testing shape: (826, 19)


In [ ]:
# Separate predictors from the ProdTaken target in both prepared splits.
X_train = train_df.drop(columns=["ProdTaken"])
y_train = train_df["ProdTaken"]

X_test = test_df.drop(columns=["ProdTaken"])
y_test = test_df["ProdTaken"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (3302, 18)
y_train: (3302,)
X_test: (826, 18)
y_test: (826,)


In [ ]:
# Detect categorical and numerical columns so each group receives suitable preprocessing.
categorical_features = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

print("\nNumber of categorical features:", len(categorical_features))
print("Number of numerical features:", len(numerical_features))

Categorical features:
['TypeofContact', 'Occupation', 'Gender', 'ProductPitched', 'MaritalStatus', 'Designation']

Numerical features:
['Age', 'CityTier', 'DurationOfPitch', 'NumberOfPersonVisiting', 'NumberOfFollowups', 'PreferredPropertyStar', 'NumberOfTrips', 'Passport', 'PitchSatisfactionScore', 'OwnCar', 'NumberOfChildrenVisiting', 'MonthlyIncome']

Number of categorical features: 6
Number of numerical features: 12


In [7]:
# Building the preprocessing pipeline for categorical and numerical features

numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [ ]:
# Store experiment metadata in the project-local SQLite database and select the shared experiment.
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

mlflow.set_experiment("wellness-tourism-experiments")

2026/09/19 22:28:12 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/19 22:28:12 INFO mlflow.store.db.utils: Updating database tables
2026/09/19 22:28:12 INFO mlflow.tracking.fluent: Experiment with name 'wellness-tourism-experiments' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/timmayabi/Projects/wellness_tourism_mlops/notebooks/mlruns/1', creation_time=1789846092410, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789846092410, lifecycle_stage='active', name='wellness-tourism-experiments', tags={}, trace_location=None, workspace='default'>

In [ ]:
# Confirm which MLflow backend will receive the experiment runs.
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: sqlite:///../mlflow.db


In [ ]:
# Couple preprocessing with a reproducible Decision Tree baseline in one inference-ready pipeline.
baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=42))
    ]
)

baseline_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3I

In [13]:
# Training the baseline model
baseline_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](18,)","['Age','TypeofContact','CityTier',...,'NumberOfChildrenVisiting', 'Designation','MonthlyIncome']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,18
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``r

In [14]:
# Making initial predictions with the baseline model
y_pred = baseline_pipeline.predict(X_test)
y_pred_proba = baseline_pipeline.predict_proba(X_test)[:, 1]

In [15]:
# Calculate evaluation metrics for the baseline model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

Accuracy : 0.8947
Precision: 0.7195
Recall   : 0.7421
F1 Score : 0.7307
ROC-AUC  : 0.8366


In [ ]:
# Inspect per-class precision, recall, and F1 to supplement the aggregate metrics.
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.93      0.93       667
           1       0.72      0.74      0.73       159

    accuracy                           0.89       826
   macro avg       0.83      0.84      0.83       826
weighted avg       0.90      0.89      0.90       826



In [20]:
# Logging the baseline model metrics to MLflow

with mlflow.start_run(run_name="DecisionTree_Baseline"):

    baseline_pipeline.fit(X_train, y_train)

    y_pred = baseline_pipeline.predict(X_test)
    y_pred_proba = baseline_pipeline.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    # Log model parameters
    mlflow.log_param("model", "DecisionTreeClassifier")
    mlflow.log_param("random_state", 42)

    # Log evaluation metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("roc_auc", roc_auc)

    # Log the complete preprocessing + model pipeline
    mlflow.sklearn.log_model(
        baseline_pipeline,
        name="model",
        serialization_format="skops",
        skops_trusted_types=[
            "numpy.dtype",
            "sklearn.tree._tree.Tree"
        ]
    )

    print("Baseline experiment logged successfully.")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")

Baseline experiment logged successfully.
Accuracy : 0.8947
Precision: 0.7195
Recall   : 0.7421
F1 Score : 0.7307
ROC-AUC  : 0.8366


In [ ]:
# Retrieve the active experiment and display where MLflow stores its metadata and artifacts.
experiment = mlflow.get_experiment_by_name(
    "wellness-tourism-experiments"
)

print("Experiment ID:", experiment.experiment_id)
print("Artifact location:", experiment.artifact_location)
print("Tracking URI:", mlflow.get_tracking_uri())

Experiment ID: 1
Artifact location: /Users/timmayabi/Projects/wellness_tourism_mlops/notebooks/mlruns/1
Tracking URI: sqlite:///../mlflow.db


In [ ]:
# Compare all recorded runs using identifiers, status, and the core classification metrics.
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

runs[[
    "run_id",
    "tags.mlflow.runName",
    "status",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1_score",
    "metrics.roc_auc"
]]

,run_id,tags.mlflow.runName,status,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc
0,99d84c6bceee4004a9a52264d9f3c55a,DecisionTree_Baseline,FINISHED,0.894673,0.719512,0.742138,0.73065,0.836586
1,54ab8b99da624a3f8e24468fd37a3cbe,DecisionTree_Baseline,FAILED,0.894673,0.719512,0.742138,0.73065,0.836586


In [24]:
# Can tuning the decision tree improve model performance?

dt_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=42))
    ]
)

In [ ]:
# Define the Decision Tree search space for complexity and split regularization.
dt_param_grid = {
    "classifier__max_depth": [3, 5, 10, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__criterion": ["gini", "entropy"]
}

In [ ]:
# Search Decision Tree configurations with five-fold cross-validation, optimizing F1 for class balance.
dt_grid_search = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=dt_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

In [ ]:
# Fit every Decision Tree candidate and retain the configuration with the strongest mean CV F1.
dt_grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 72 candidates, totalling 360 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__criterion': ['gini', 'entropy'], 'classifier__max_depth': [3, 5, ...], 'classifier__min_samples_leaf': [1, 2, ...], 'classifier__min_samples_split': [2, 5, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about mult

In [ ]:
# Report the winning Decision Tree configuration and its cross-validation score.
print("Best parameters:")
print(dt_grid_search.best_params_)

print("\nBest cross-validation F1:")
print(dt_grid_search.best_score_)

Best parameters:
{'classifier__criterion': 'gini', 'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2}

Best cross-validation F1:
0.6949050697667111


In [ ]:
# Evaluate the tuned Decision Tree once on the held-out test set.
best_dt_model = dt_grid_search.best_estimator_

y_pred_tuned = best_dt_model.predict(X_test)
y_pred_proba_tuned = best_dt_model.predict_proba(X_test)[:, 1]

accuracy_tuned = accuracy_score(y_test, y_pred_tuned)
precision_tuned = precision_score(y_test, y_pred_tuned)
recall_tuned = recall_score(y_test, y_pred_tuned)
f1_tuned = f1_score(y_test, y_pred_tuned)
roc_auc_tuned = roc_auc_score(y_test, y_pred_proba_tuned)

print(f"Accuracy : {accuracy_tuned:.4f}")
print(f"Precision: {precision_tuned:.4f}")
print(f"Recall   : {recall_tuned:.4f}")
print(f"F1 Score : {f1_tuned:.4f}")
print(f"ROC-AUC  : {roc_auc_tuned:.4f}")

Accuracy : 0.8947
Precision: 0.7195
Recall   : 0.7421
F1 Score : 0.7307
ROC-AUC  : 0.8366


In [30]:
with mlflow.start_run(run_name="DecisionTree_Tuned"):

    # Log best hyperparameters
    mlflow.log_param("model", "DecisionTreeClassifier")
    
    for param, value in dt_grid_search.best_params_.items():
        mlflow.log_param(param, value)

    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("scoring", "f1")

    # Log cross-validation result
    mlflow.log_metric(
        "best_cv_f1",
        dt_grid_search.best_score_
    )

    # Log test metrics
    mlflow.log_metric("accuracy", accuracy_tuned)
    mlflow.log_metric("precision", precision_tuned)
    mlflow.log_metric("recall", recall_tuned)
    mlflow.log_metric("f1_score", f1_tuned)
    mlflow.log_metric("roc_auc", roc_auc_tuned)

    # Log complete tuned pipeline
    mlflow.sklearn.log_model(
        best_dt_model,
        name="model",
        serialization_format="skops",
        skops_trusted_types=[
            "numpy.dtype",
            "sklearn.tree._tree.Tree"
        ]
    )

    print("Tuned Decision Tree logged successfully.")

Tuned Decision Tree logged successfully.


In [ ]:
# Build a Random Forest pipeline that applies the same preprocessing used by other models.
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            random_state=42,
            n_jobs=-1
        ))
    ]
)

In [ ]:
# Define candidate forest sizes and regularization settings for Random Forest tuning.
rf_param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2]
}

In [ ]:
# Re-establish the Random Forest search grid used by the following cross-validation step.
rf_param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2]
}

In [ ]:
# Tune the Random Forest with five-fold cross-validation using F1 as the selection metric.
rf_grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

In [ ]:
# Fit all Random Forest candidates and select the model with the best mean CV F1.
rf_grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__max_depth': [None, 10, ...], 'classifier__min_samples_leaf': [1, 2], 'classifier__min_samples_split': [2, 5], 'classifier__n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metrice

In [ ]:
# Display the selected Random Forest hyperparameters and validation performance.
print("Best parameters:")
print(rf_grid_search.best_params_)

print("\nBest cross-validation F1:")
print(rf_grid_search.best_score_)

Best parameters:
{'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200}

Best cross-validation F1:
0.6911393984295717


In [ ]:
# Measure the tuned Random Forest on untouched test data using class and probability metrics.
best_rf_model = rf_grid_search.best_estimator_

y_pred_rf = best_rf_model.predict(X_test)
y_pred_proba_rf = best_rf_model.predict_proba(X_test)[:, 1]

accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)

print(f"Accuracy : {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall   : {recall_rf:.4f}")
print(f"F1 Score : {f1_rf:.4f}")
print(f"ROC-AUC  : {roc_auc_rf:.4f}")

Accuracy : 0.9080
Precision: 0.9192
Recall   : 0.5723
F1 Score : 0.7054
ROC-AUC  : 0.9665


In [ ]:
# Restate the best Random Forest search result before experiment logging.
print("Best parameters:")
print(rf_grid_search.best_params_)

print("\nBest CV F1:")
print(rf_grid_search.best_score_)

Best parameters:
{'classifier__max_depth': None, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200}

Best CV F1:
0.6911393984295717


In [48]:
with mlflow.start_run(run_name="RandomForest_Tuned"):

    # Model information
    mlflow.log_param("model", "RandomForestClassifier")

    # Best parameters selected by GridSearchCV
    for param, value in rf_grid_search.best_params_.items():
        mlflow.log_param(param, value)

    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("scoring", "f1")

    # Cross-validation metric
    mlflow.log_metric(
        "best_cv_f1",
        rf_grid_search.best_score_
    )

    # Test metrics
    mlflow.log_metric("accuracy", accuracy_rf)
    mlflow.log_metric("precision", precision_rf)
    mlflow.log_metric("recall", recall_rf)
    mlflow.log_metric("f1_score", f1_rf)
    mlflow.log_metric("roc_auc", roc_auc_rf)

    # Save the complete preprocessing + RF pipeline
    mlflow.sklearn.log_model(
        best_rf_model,
        name="model",
        serialization_format="skops",
        skops_trusted_types=[
            "numpy.dtype",
            "sklearn.tree._tree.Tree"
        ]
    )

    print("Random Forest experiment logged successfully.")

Random Forest experiment logged successfully.


In [ ]:
# Build a Gradient Boosting pipeline with the shared preprocessing contract.
gb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", GradientBoostingClassifier(
            random_state=42
        ))
    ]
)

In [ ]:
# Define Gradient Boosting candidates across ensemble size, learning rate, and tree complexity.
gb_param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__max_depth": [2, 3, 5],
    "classifier__min_samples_split": [2, 5]
}

In [ ]:
# Search Gradient Boosting configurations with the same F1-based validation protocol.
gb_grid_search = GridSearchCV(
    estimator=gb_pipeline,
    param_grid=gb_param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

In [ ]:
# Fit all Gradient Boosting candidates and retain the highest-scoring CV model.
gb_grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__learning_rate': [0.05, 0.1], 'classifier__max_depth': [2, 3, ...], 'classifier__min_samples_split': [2, 5], 'classifier__n_estimators': [100, 200]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metriceva

In [ ]:
# Review the selected Gradient Boosting settings and their mean validation F1.
print("Best parameters:")
print(gb_grid_search.best_params_)

print("\nBest cross-validation F1:")
print(gb_grid_search.best_score_)

Best parameters:
{'classifier__learning_rate': 0.1, 'classifier__max_depth': 5, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200}

Best cross-validation F1:
0.7480768079912118


In [ ]:
# Evaluate the tuned Gradient Boosting model on held-out labels and probabilities.
best_gb_model = gb_grid_search.best_estimator_

y_pred_gb = best_gb_model.predict(X_test)
y_pred_proba_gb = best_gb_model.predict_proba(X_test)[:, 1]

accuracy_gb = accuracy_score(y_test, y_pred_gb)
precision_gb = precision_score(y_test, y_pred_gb)
recall_gb = recall_score(y_test, y_pred_gb)
f1_gb = f1_score(y_test, y_pred_gb)
roc_auc_gb = roc_auc_score(y_test, y_pred_proba_gb)

print(f"Accuracy : {accuracy_gb:.4f}")
print(f"Precision: {precision_gb:.4f}")
print(f"Recall   : {recall_gb:.4f}")
print(f"F1 Score : {f1_gb:.4f}")
print(f"ROC-AUC  : {roc_auc_gb:.4f}")

Accuracy : 0.9274
Precision: 0.9160
Recall   : 0.6855
F1 Score : 0.7842
ROC-AUC  : 0.9495


In [ ]:
# Restate the winning Gradient Boosting configuration before logging the run.
print("Best parameters:")
print(gb_grid_search.best_params_)

print("\nBest CV F1:")
print(gb_grid_search.best_score_)

Best parameters:
{'classifier__learning_rate': 0.1, 'classifier__max_depth': 5, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200}

Best CV F1:
0.7480768079912118


In [ ]:
# Persist the tuned Gradient Boosting configuration, performance, and complete pipeline in MLflow.
with mlflow.start_run(run_name="GradientBoosting_Tuned"):

    mlflow.log_param("model", "GradientBoostingClassifier")

    for param, value in gb_grid_search.best_params_.items():
        mlflow.log_param(param, value)

    mlflow.log_param("cv_folds", 5)
    mlflow.log_param("scoring", "f1")

    mlflow.log_metric(
        "best_cv_f1",
        gb_grid_search.best_score_
    )

    mlflow.log_metric("accuracy", accuracy_gb)
    mlflow.log_metric("precision", precision_gb)
    mlflow.log_metric("recall", recall_gb)
    mlflow.log_metric("f1_score", f1_gb)
    mlflow.log_metric("roc_auc", roc_auc_gb)

    mlflow.sklearn.log_model(
        best_gb_model,
        name="model",
        serialization_format="skops",
        skops_trusted_types=[
            "numpy.dtype",
            "sklearn.tree._tree.Tree"
        ]
    )

    print("Gradient Boosting experiment logged successfully.")

Gradient Boosting experiment logged successfully.


In [ ]:
# Assemble a common test-metric leaderboard and rank models by F1 score.
results = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting"
    ],
    "Accuracy": [
        accuracy_tuned,
        accuracy_rf,
        accuracy_gb
    ],
    "Precision": [
        precision_tuned,
        precision_rf,
        precision_gb
    ],
    "Recall": [
        recall_tuned,
        recall_rf,
        recall_gb
    ],
    "F1 Score": [
        f1_tuned,
        f1_rf,
        f1_gb
    ],
    "ROC-AUC": [
        roc_auc_tuned,
        roc_auc_rf,
        roc_auc_gb
    ]
})

results.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Gradient Boosting,0.927361,0.915966,0.685535,0.784173,0.949516
1,Decision Tree,0.894673,0.719512,0.742138,0.730650,0.836586
2,Random Forest,0.907990,0.919192,0.572327,0.705426,0.966479


## Model Selection

The models were compared using Accuracy, Precision, Recall, F1 Score, and
ROC-AUC.

Because the target variable is imbalanced, F1 Score was used as the primary
model-selection metric rather than accuracy alone.

Gradient Boosting achieved the highest test F1 Score (0.7842), while also
maintaining high accuracy (0.9274), precision (0.9160), and ROC-AUC (0.9495).

Therefore, Gradient Boosting was selected as the final model for deployment.

In [ ]:
# Serialize the selected Gradient Boosting pipeline locally in the safer skops format.
os.makedirs("../model", exist_ok=True)

model_path = "../model/wellness_tourism_model.skops"

sio.dump(best_gb_model, model_path)

print(f"Model saved to {model_path}")

Model saved to ../model/wellness_tourism_model.skops


In [ ]:
# Inspect serialized object types that skops requires the caller to trust explicitly.
unknown_types = sio.get_untrusted_types(
    file=model_path
)

print(unknown_types)

['numpy.dtype', 'sklearn.tree._tree.Tree']


In [ ]:
# Reload the local artifact while allowing only the types identified during inspection.
loaded_model = sio.load(
    model_path,
    trusted=unknown_types
)

In [ ]:
# Run a smoke test to confirm the reloaded pipeline can predict the full test split.
test_predictions = loaded_model.predict(X_test)

print("Model loaded successfully.")
print("Number of predictions:", len(test_predictions))

Model loaded successfully.
Number of predictions: 826


In [ ]:
# Initialize the Hugging Face API client and identify the destination model repository.
api = HfApi()
model_repo_id = "motidev/wellness-tourism-model"

In [ ]:
# Publish the serialized pipeline to the Hugging Face model repository for deployment.
api.upload_file(
    path_or_fileobj=model_path,
    path_in_repo="wellness_tourism_model.skops",
    repo_id=model_repo_id,
    repo_type="model"
)

print("Model uploaded successfully to Hugging Face Model Hub.")

Processing Files (1 / 1): 100%|██████████| 7.34MB / 7.34MB,  588kB/s  
New Data Upload: 100%|██████████| 7.34MB / 7.34MB,  588kB/s  


Model uploaded successfully to Hugging Face Model Hub.


In [71]:
model_card = """
# Wellness Tourism Purchase Prediction Model

## Model Description

This model predicts whether a customer is likely to purchase a Wellness
Tourism Package.

## Target

`ProdTaken`

- 0: Customer did not purchase the package
- 1: Customer purchased the package

## Model

Gradient Boosting Classifier with a preprocessing pipeline for numerical and
categorical features.

## Model Selection

Decision Tree, Random Forest, and Gradient Boosting models were evaluated.

F1 Score was used as the primary model-selection metric because the target
variable is imbalanced.

## Test Performance

| Metric | Score |
|---|---:|
| Accuracy | 0.9274 |
| Precision | 0.9160 |
| Recall | 0.6855 |
| F1 Score | 0.7842 |
| ROC-AUC | 0.9495 |

## Experiment Tracking

Model experiments and hyperparameters were tracked using MLflow.

## Intended Use

The model is intended to demonstrate an end-to-end MLOps workflow for
predicting customer interest in a Wellness Tourism Package.
"""

with open("../model/README.md", "w") as f:
    f.write(model_card)

In [ ]:
# Download the published artifact to verify the exact model available to consumers.
downloaded_model_path = hf_hub_download(
    repo_id=model_repo_id,
    filename="wellness_tourism_model.skops"
)

print(downloaded_model_path)

/Users/timmayabi/.cache/huggingface/hub/models--motidev--wellness-tourism-model/snapshots/1b9b2c4706ffdc68783467e6b644cd3bba0633d1/wellness_tourism_model.skops


In [ ]:
# Safely load the downloaded artifact and confirm it performs inference on the test schema.
unknown_types = sio.get_untrusted_types(
    file=downloaded_model_path
)

hf_model = sio.load(
    downloaded_model_path,
    trusted=unknown_types
)

hf_predictions = hf_model.predict(X_test)

print("Hugging Face model loaded successfully.")
print("Predictions:", len(hf_predictions))

Hugging Face model loaded successfully.
Predictions: 826
